In [1]:
import subprocess
import pandas as pd
import os

# Define a helper function to run scripts and display output
def run_script(script_path, as_module=False):
    print(f"\n--- Running {script_path} (as_module={as_module}) ---")
    if as_module:
        # Convert file path to module path (e.g., scripts/features/script.py -> scripts.features.script)
        module_path = script_path.replace(os.sep, '.').replace('.py', '')
        command = ['python3', '-m', module_path]
    else:
        command = ['python3', script_path]

    try:
        result = subprocess.run(command, capture_output=True, text=True, check=True)
        print(result.stdout)
        if result.stderr:
            print(f"[ERROR/WARNING from {script_path}]\n{result.stderr}")
    except subprocess.CalledProcessError as e:
        print(f"Script {script_path} failed with error code {e.returncode}\nError Output:\n{e.stderr}")
    except FileNotFoundError:
        print(f"Error: Python3 executable not found or script not at {script_path}")
    print(f"--- Finished {script_path} ---\n")

# Define paths (assuming notebook is in the project root)
SCRIPTS_DIR = 'scripts/'
DATASOURCE_PROCESSED_DIR = 'datasource/processed/'
DATASOURCE_RAW_DIR = 'datasource/raw/'
MODELS_DIR = 'output/models/' # Added for consistency

# Ensure output directories exist
os.makedirs(DATASOURCE_PROCESSED_DIR, exist_ok=True)
os.makedirs(DATASOURCE_RAW_DIR, exist_ok=True)
os.makedirs(MODELS_DIR, exist_ok=True)



In [2]:
run_script(os.path.join(SCRIPTS_DIR, 'process/merge_transaction_sets.py'), as_module=True)
run_script(os.path.join(SCRIPTS_DIR, 'process/merge_mixer_recipient_txns.py'), as_module=True)
run_script(os.path.join(SCRIPTS_DIR, 'process/save_wallet_reference.py'), as_module=True)
run_script(os.path.join(SCRIPTS_DIR, 'process/validate_merged_data.py'), as_module=True)

print("\n--- Preview of all_transactions_labeled.csv (first 5 rows) ---")
try:
    df_raw_merged = pd.read_csv(os.path.join(DATASOURCE_RAW_DIR, 'all_transactions_labeled.csv'))
    print(df_raw_merged.head())
except FileNotFoundError:
    print("all_transactions_labeled.csv not found. Ensure fetch and merge scripts ran successfully.")




--- Running scripts/process/merge_transaction_sets.py (as_module=True) ---
Merging datasets...
Removed 120545 duplicate transactions based on hash.
Wallet risk labels saved to: output/models/wallet_risk_labels.joblib
Global gas 95th percentile saved to: output/models/global_gas_95th_percentile.joblib
Merged 254454 unique transactions into datasource/raw/all_transactions_labeled.csv

[ERROR/WARNING from scripts/process/merge_transaction_sets.py]
/Users/jzackslineandreela/Downloads/noir-framework/scripts/process/merge_transaction_sets.py:15: DtypeWarning: Columns (8,14) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(path)

--- Finished scripts/process/merge_transaction_sets.py ---


--- Running scripts/process/merge_mixer_recipient_txns.py (as_module=True) ---
Merged mixer-recipient txns into: datasource/raw/all_transactions_labeled.csv

[ERROR/WARNING from scripts/process/merge_mixer_recipient_txns.py]
/Users/jzackslineandreela/Downloads/no

/var/folders/s_/6468mz_94k1g_vytp8czgb4h0000gn/T/ipykernel_72825/2777268634.py:8: DtypeWarning: Columns (13,20,21) have mixed types. Specify dtype option on import or set low_memory=False.
  df_raw_merged = pd.read_csv(os.path.join(DATASOURCE_RAW_DIR, 'all_transactions_labeled.csv'))


In [3]:
run_script(os.path.join(SCRIPTS_DIR, 'features/build_features_l0_aggregate.py'), as_module=True)

print("\n--- Preview of features_l0_aggregate.csv (first 5 rows) ---")
try:
    df_l0 = pd.read_csv(os.path.join(DATASOURCE_PROCESSED_DIR, 'features_l0_aggregate.csv'))
    print(df_l0.head())
except FileNotFoundError:
    print("features_l0_aggregate.csv not found.")




--- Running scripts/features/build_features_l0_aggregate.py (as_module=True) ---
 Wallet features saved to: datasource/processed/features_l0_aggregate.csv

[ERROR/WARNING from scripts/features/build_features_l0_aggregate.py]
/Users/jzackslineandreela/Downloads/noir-framework/scripts/features/build_features_l0_aggregate.py:25: DtypeWarning: Columns (13,20,21) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(INPUT_FILE)

--- Finished scripts/features/build_features_l0_aggregate.py ---


--- Preview of features_l0_aggregate.csv (first 5 rows) ---
                               wallet_address  total_transactions  \
0  0x000000000004444c5dc75cb358380d2e3de08a90                 446   
1  0x00000000219ab540356cbb839cbe05303d7705fa               10000   
2  0x0017408209fd546fda8f5f5135ebfb346a7cfabf                  34   
3  0x0039f22efb07a647557c7c5d17854cfd6d489ef3                1876   
4  0x0100dc5672f702e705fc693218a3ad38fed6553d               

In [4]:
run_script(os.path.join(SCRIPTS_DIR, 'features/build_features_l1_behavior.py'), as_module=True)

print("\n--- Preview of features_l1_behavior.csv (first 5 rows) ---")
try:
    df_l1 = pd.read_csv(os.path.join(DATASOURCE_PROCESSED_DIR, 'features_l1_behavior.csv'))
    print(df_l1.head())
except FileNotFoundError:
    print("features_l1_behavior.csv not found.")




--- Running scripts/features/build_features_l1_behavior.py (as_module=True) ---
L1 behavioral features saved to datasource/processed/features_l1_behavior.csv

[ERROR/WARNING from scripts/features/build_features_l1_behavior.py]
/Users/jzackslineandreela/Downloads/noir-framework/scripts/features/build_features_l1_behavior.py:14: DtypeWarning: Columns (13,20,21) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(TX_PATH)

--- Finished scripts/features/build_features_l1_behavior.py ---


--- Preview of features_l1_behavior.csv (first 5 rows) ---
                               wallet_address  total_transactions  \
0  0x000000000004444c5dc75cb358380d2e3de08a90                 446   
1  0x00000000219ab540356cbb839cbe05303d7705fa               10000   
2  0x0017408209fd546fda8f5f5135ebfb346a7cfabf                  34   
3  0x0039f22efb07a647557c7c5d17854cfd6d489ef3                1876   
4  0x0100dc5672f702e705fc693218a3ad38fed6553d                  4

In [5]:
run_script(os.path.join(SCRIPTS_DIR, 'features/build_features_l2_riskflags.py'), as_module=True)

print("\n--- Preview of features_l2_riskflags.csv (first 5 rows) ---")
try:
    df_l2 = pd.read_csv(os.path.join(DATASOURCE_PROCESSED_DIR, 'features_l2_riskflags.csv'))
    print(df_l2.head())
except FileNotFoundError:
    print("features_l2_riskflags.csv not found.")




--- Running scripts/features/build_features_l2_riskflags.py (as_module=True) ---
 L2 counterparty and risk flags saved to: datasource/processed/features_l2_riskflags.csv

[ERROR/WARNING from scripts/features/build_features_l2_riskflags.py]
/Users/jzackslineandreela/Downloads/noir-framework/scripts/features/build_features_l2_riskflags.py:17: DtypeWarning: Columns (13,20,21) have mixed types. Specify dtype option on import or set low_memory=False.
  df_tx = pd.read_csv(TRANSACTIONS_FILE)

--- Finished scripts/features/build_features_l2_riskflags.py ---


--- Preview of features_l2_riskflags.csv (first 5 rows) ---
                               wallet_address  total_transactions  \
0  0x000000000004444c5dc75cb358380d2e3de08a90                 446   
1  0x00000000219ab540356cbb839cbe05303d7705fa               10000   
2  0x0017408209fd546fda8f5f5135ebfb346a7cfabf                  34   
3  0x0039f22efb07a647557c7c5d17854cfd6d489ef3                1876   
4  0x0100dc5672f702e705fc693218a3ad

In [6]:
run_script(os.path.join(SCRIPTS_DIR, 'features/build_features_l3_metaai.py'), as_module=True)

print("\n--- Preview of features_l3_metaai.csv (first 5 rows) ---")
try:
    df_l3_metaai = pd.read_csv(os.path.join(DATASOURCE_PROCESSED_DIR, 'features_l3_metaai.csv'))
    print(df_l3_metaai.head())
except FileNotFoundError:
    print("features_l3_metaai.csv not found.")




--- Running scripts/features/build_features_l3_metaai.py (as_module=True) ---
L3 Meta-AI feature columns saved to: output/models/l3_metaai_feature_columns.joblib
Attempting to save L3 Meta-AI models to /Users/jzackslineandreela/Downloads/noir-framework/output/models
L3 Meta-AI models and scaler saved to: /Users/jzackslineandreela/Downloads/noir-framework/output/models
L3 meta-AI features saved to: datasource/processed/features_l3_metaai.csv

--- Finished scripts/features/build_features_l3_metaai.py ---


--- Preview of features_l3_metaai.csv (first 5 rows) ---
                               wallet_address  total_transactions  \
0  0x000000000004444c5dc75cb358380d2e3de08a90                 446   
1  0x00000000219ab540356cbb839cbe05303d7705fa               10000   
2  0x0017408209fd546fda8f5f5135ebfb346a7cfabf                  34   
3  0x0039f22efb07a647557c7c5d17854cfd6d489ef3                1876   
4  0x0100dc5672f702e705fc693218a3ad38fed6553d                  45   

   wallet_age_day

In [7]:
run_script(os.path.join(SCRIPTS_DIR, 'features/build_features_l3_xai_tags.py'), as_module=True)

print("\n--- Preview of features_l3_metaai_xai.csv (first 5 rows) ---")
try:
    df_l3_xai = pd.read_csv(os.path.join(DATASOURCE_PROCESSED_DIR, 'features_l3_metaai_xai.csv'))
    print(df_l3_xai[['wallet_address', 'xai_reason_code', 'xai_flag']].head())
except FileNotFoundError:
    print("features_l3_metaai_xai.csv not found.")




--- Running scripts/features/build_features_l3_xai_tags.py (as_module=True) ---
L3 MetaAI + XAI tagging complete → features_l3_metaai_xai.csv

--- Finished scripts/features/build_features_l3_xai_tags.py ---


--- Preview of features_l3_metaai_xai.csv (first 5 rows) ---
                               wallet_address  \
0  0x000000000004444c5dc75cb358380d2e3de08a90   
1  0x00000000219ab540356cbb839cbe05303d7705fa   
2  0x0017408209fd546fda8f5f5135ebfb346a7cfabf   
3  0x0039f22efb07a647557c7c5d17854cfd6d489ef3   
4  0x0100dc5672f702e705fc693218a3ad38fed6553d   

                             xai_reason_code  xai_flag  
0            model_anomaly|high_failure_rate         1  
1                                   burst_tx         1  
2                           dormant_awakened         1  
3  dormant_awakened|fraud_link|model_anomaly         1  
4                                      clean         0  


In [8]:
run_script(os.path.join(SCRIPTS_DIR, 'process/prepare_features_for_training.py'), as_module=True)




--- Running scripts/process/prepare_features_for_training.py (as_module=True) ---
Dropped 1 highly correlated features: ['txn_span_hours']
Saved cleaned feature dataset to: datasource/processed/features_for_training.csv

--- Finished scripts/process/prepare_features_for_training.py ---



In [9]:
run_script(os.path.join(SCRIPTS_DIR, 'models/train_supervised_models.py'), as_module=True)




--- Running scripts/models/train_supervised_models.py (as_module=True) ---
Model feature columns saved to: output/models/model_feature_columns.joblib

 Training Random Forest...
Random Forest model saved to: output/models/random_forest_model.joblib
 Training XGBoost...
XGBoost model saved to: output/models/xgboost_model.joblib

 Random Forest Classification Report:
              precision    recall  f1-score   support

           0       0.83      1.00      0.91        10
           1       0.91      0.94      0.92        32
           2       1.00      0.57      0.73         7

    accuracy                           0.90        49
   macro avg       0.91      0.84      0.85        49
weighted avg       0.91      0.90      0.89        49


Confusion Matrix:
[[10  0  0]
 [ 2 30  0]
 [ 0  3  4]]

 XGBoost Classification Report:
              precision    recall  f1-score   support

           0       0.67      1.00      0.80        10
           1       0.90      0.84      0.87        3

In [10]:
run_script(os.path.join(SCRIPTS_DIR, 'gnn/build_graph_dataset.py'), as_module=True)
run_script(os.path.join(SCRIPTS_DIR, 'gnn/prep_gnn_input.py'), as_module=True)
run_script(os.path.join(SCRIPTS_DIR, 'gnn/train_gnn_model.py'), as_module=True)
run_script(os.path.join(SCRIPTS_DIR, 'gnn/analyse_predictions.py'), as_module=True)




--- Running scripts/gnn/build_graph_dataset.py (as_module=True) ---
Clipped edge weights above 1e6 ETH
 Edge list saved to output/gnn/graph_edgelist.csv
Dropped zero-variance columns: ['num_normal_counterparties', 'circular_flow_flag', 'gas_anomaly_flag', 'rapid_bridging_flag', 'smart_contract_misuse_flag', 'mixer_then_bridge_flag', 'mixer_exit_tx_count', 'same_recipient_ratio', 'smart_contract_failures', 'layer_hopping_count', 'circular_tx_ratio', 'avg_gas_fee']
 Node features saved to output/gnn/graph_node_features.csv
 Graph saved to output/gnn/graph.gpickle

--- Finished scripts/gnn/build_graph_dataset.py ---


--- Running scripts/gnn/prep_gnn_input.py (as_module=True) ---
 GNN input saved to: output/gnn/gnn_data.pt
index_to_wallet mapping saved to output/gnn/index_to_wallet.pkl

--- Finished scripts/gnn/prep_gnn_input.py ---


--- Running scripts/gnn/train_gnn_model.py (as_module=True) ---
⚠️ Unmatched wallets: 58984 (will be skipped)
Initial node feature shape: torch.Size([59221

In [11]:
run_script(os.path.join(SCRIPTS_DIR, 'stats/transaction_stats_summary.py'), as_module=True)
run_script(os.path.join(SCRIPTS_DIR, 'stats/analyze_feature_stats.py'), as_module=True)
run_script(os.path.join(SCRIPTS_DIR, 'stats/analyse_graphdata.py'), as_module=True)
run_script(os.path.join(SCRIPTS_DIR, 'phase2/analyze_feature_flags.py'), as_module=True)




--- Running scripts/stats/transaction_stats_summary.py (as_module=True) ---
Reading fraud: datasource/raw/fraud_transactions.csv
Reading normal: datasource/raw/normal_transactions.csv
Reading mixer: datasource/raw/mixer_interactions.csv

 Saved summary to: output/transaction_summary.csv

[ERROR/WARNING from scripts/stats/transaction_stats_summary.py]
/Users/jzackslineandreela/Downloads/noir-framework/scripts/stats/transaction_stats_summary.py:29: DtypeWarning: Columns (8,14) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(path)

--- Finished scripts/stats/transaction_stats_summary.py ---


--- Running scripts/stats/analyze_feature_stats.py (as_module=True) ---
 Feature analysis complete.
Missing values in: []
Low variance features: ['num_normal_counterparties', 'circular_flow_flag', 'gas_anomaly_flag', 'rapid_bridging_flag', 'smart_contract_misuse_flag', 'mixer_then_bridge_flag', 'mixer_exit_tx_count', 'same_recipient_ratio', 'smart_contrac